# BI Intern Assignment — D2C Analytics Challenge

This notebook contains solutions to the 3 SQL questions, the Python task, and the LLM dashboard prompt + link.

## Setup
Load the 4 datasets from the provided Excel file and load them into an in-memory SQLite database so the SQL tasks can be run as real SQL.

In [ ]:
import pandas as pd
import sqlite3

# Change this path if the assignment file is named differently / in a different folder
EXCEL_PATH = "bi_intern_assignment.xlsx"

orders    = pd.read_excel(EXCEL_PATH, sheet_name="📊 orders")
customers = pd.read_excel(EXCEL_PATH, sheet_name="👤 customers")
catalog   = pd.read_excel(EXCEL_PATH, sheet_name="🗂 product_catalog")
pricing   = pd.read_excel(EXCEL_PATH, sheet_name="💰 product_pricing")

print(orders.shape, customers.shape, catalog.shape, pricing.shape)
orders.head()

In [ ]:
# Load into SQLite so we can write real, runnable SQL
conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
catalog.to_sql("product_catalog", conn, index=False, if_exists="replace")
pricing.to_sql("product_pricing", conn, index=False, if_exists="replace")
print("Tables loaded:", ["orders", "customers", "product_catalog", "product_pricing"])

---
## TASK 1 — SQL Analysis

### QS1 — Orders & Avg MRP by Customer Segment

**Question:** For each `customer_segment` (Gold / Silver / Bronze) calculate total_orders and avg_MRP, ordered by total_orders DESC.

**Approach:** Join `orders` to `customers` on `customer_id`, group by `customer_segment`, and aggregate.

In [ ]:
q1 = '''
SELECT
    c.customer_segment,
    COUNT(o.order_id)      AS total_orders,
    ROUND(AVG(o.MRP), 2)   AS avg_MRP
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_segment
ORDER BY total_orders DESC;
'''
q1_result = pd.read_sql(q1, conn)
q1_result

### QS2 — Total Revenue by Product

**Question:** For each `product_name` calculate total_revenue.

**Approach:** `product_pricing` gives a channel-specific `discount_pct` for each product, so revenue can't be computed from `orders` alone. Revenue per order line = `MRP * quantity * (1 - discount_pct / 100)`. Join `orders` to `product_pricing` on **both** `product_name` and `channel` (since the discount depends on the channel the order came through), then sum per product.

In [ ]:
q2 = '''
SELECT
    o.product_name,
    ROUND(SUM(o.MRP * o.quantity * (1 - p.discount_pct / 100.0)), 2) AS total_revenue
FROM orders o
JOIN product_pricing p
    ON o.product_name = p.product_name
   AND o.channel      = p.channel
GROUP BY o.product_name
ORDER BY total_revenue DESC;
'''
q2_result = pd.read_sql(q2, conn)
q2_result

### QS3 — Channel with the Highest Discount

**Question:** As per orders made, which channel has given the highest discount_pct?

**Approach:** Join `orders` to `product_pricing` on product + channel (same as QS2) to get the discount_pct actually applied on each order line, then average by channel.

In [ ]:
q3 = '''
SELECT
    o.channel,
    ROUND(AVG(p.discount_pct), 2) AS avg_discount_pct
FROM orders o
JOIN product_pricing p
    ON o.product_name = p.product_name
   AND o.channel      = p.channel
GROUP BY o.channel
ORDER BY avg_discount_pct DESC;
'''
q3_result = pd.read_sql(q3, conn)
q3_result

In [ ]:
top_channel = q3_result.iloc[0]
print(f"Answer: '{top_channel["channel"]}' has given the highest average discount at "
      f"{top_channel["avg_discount_pct"]}%.")

---
## TASK 2 — Python Analysis

### QP1 — Category Performance Summary

**Task:** Build a dataframe with columns: `Category | Total Revenue | Orders | Avg Order Value`.

**Approach:** `orders` already has a `category` column, so no need to join `product_catalog` for this. Join `orders` to `product_pricing` (on product_name + channel) to get the applicable discount, compute per-line revenue, then group by category.

In [ ]:
df = orders.merge(pricing, on=["product_name", "channel"], how="left")
df["revenue"] = df["MRP"] * df["quantity"] * (1 - df["discount_pct"] / 100)

category_summary = (
    df.groupby("category")
      .agg(**{
          "Total Revenue": ("revenue", "sum"),
          "Orders": ("order_id", "count"),
      })
      .reset_index()
      .rename(columns={"category": "Category"})
)
category_summary["Avg Order Value"] = (
    category_summary["Total Revenue"] / category_summary["Orders"]
).round(2)
category_summary["Total Revenue"] = category_summary["Total Revenue"].round(2)
category_summary = category_summary.sort_values("Total Revenue", ascending=False).reset_index(drop=True)

category_summary

In [ ]:
# Quick business takeaway
top_cat = category_summary.iloc[0]
print(f"Insight: '{top_cat["Category"]}' is the top-performing category, generating "
      f"₹{top_cat["Total Revenue"]:,.0f} in revenue across {top_cat["Orders"]} orders "
      f"(Avg Order Value ₹{top_cat["Avg Order Value"]:,.0f}).")

---
## TASK 3 — LLM-Powered HTML Dashboard

### Initial prompt given to the LLM

> Build a single-page HTML dashboard for a D2C brand's channel and product performance. It must run in any browser with no server or build step — one self-contained .html file, using Chart.js from a CDN for the charts and no other dependencies.
>
> Show:
> 1. Four KPI cards at the top: Total Revenue, Total Orders, Avg Order Value, Units Sold.
> 2. A bar chart of revenue by channel.
> 3. A doughnut chart of revenue share by category.
> 4. A horizontal bar chart of the top 10 products by revenue.
> 5. A table of channel performance (orders, revenue, and a visual bar indicator).
>
> Use a clean dark theme, rounded cards, and make it responsive on mobile. Hardcode the aggregated data (from the orders/pricing tables, revenue = MRP × quantity × (1 - discount_pct/100)) directly into the HTML/JS — no external data fetch needed.

### Output
The generated dashboard is saved as **`dashboard.html`** in the same folder as this notebook. Open it directly in any browser — no server needed.

**Dashboard link:** `dashboard.html` (see submission folder / attached file)

In [ ]:
# Optional: open the dashboard directly from the notebook to preview it
from IPython.display import IFrame
IFrame(src="dashboard.html", width="100%", height=800)

---
## Summary

- **QS1:** Gold segment has the most orders (22) with the highest avg MRP.
- **QS2:** Vitamin C Serum is the top revenue-generating product (~₹30,730).
- **QS3:** Amazon gives the highest average discount (~17.4%).
- **QP1:** Wellness and Skincare together drive ~70% of total category revenue.
- **Dashboard:** `dashboard.html` — single-file, no-server HTML dashboard covering channel + product performance.